In [1]:
# Imports and setup
import json
import warnings
import torch
from pprint import pprint

import numpy as np
import pandas as pd

from benchmark.config import DATA_PROCESSED, N_TRIALS, NUM_FOLDS
from benchmark.models.xgboost_model import (
    build_pipeline_reg,
    build_preprocessor,
    evaluate_reg,
    export_results_reg,
    final_fit_reg,
    get_search_space,
    run_search_reg,
)

warnings.filterwarnings("ignore")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Main config
data_name = "diamonds"
processed = DATA_PROCESSED / data_name
k_folds = NUM_FOLDS
n_trials = N_TRIALS

xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
# xgb_device = "cpu"
print("XGBoost device:", xgb_device)

XGBoost device: cuda


In [3]:
# Load data and metadata
train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as file:
    metadata = json.load(file)

print("Dataset path:", processed)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nMetadata:")
pprint(metadata)

Dataset path: /home/shado/college_project/comp9417/9417GroupProject/data/processed/diamonds
Train shape: (43152, 11)
Test shape: (10788, 11)

Metadata:
{'cat_cols': ['cut', 'color', 'clarity'],
 'num_cols': ['carat', 'depth', 'table', 'x', 'y', 'z', 'volumn'],
 'target_col': 'price'}


In [4]:
# Split x and y
target_column = metadata["target_col"]

x_train_full = train_df.drop(columns=[target_column]).copy()
y_train_full = train_df[target_column].to_numpy(dtype=np.float32)

x_test = test_df.drop(columns=[target_column]).copy()
y_test = test_df[target_column].to_numpy(dtype=np.float32)

In [5]:
# Build preprocessor and run hyperparameter search
preprocessor = build_preprocessor(metadata)
pipeline = build_pipeline_reg(preprocessor, device=xgb_device, n_jobs=1)
search_space = get_search_space()

best_search_params, best_model_params, best_cv_mse, search_time = run_search_reg(
    pipeline,
    search_space,
    x_train_full,
    y_train_full,
    k_folds=k_folds,
    n_trials=n_trials,
)

print("\nBest CV MSE:", best_cv_mse)
print("\nBest hyperparameters:")
pprint(best_model_params)


Best CV MSE: 323749.2375

Best hyperparameters:
{'colsample_bytree': 0.7936329675035503,
 'gamma': 1.030704125772499e-08,
 'learning_rate': 0.06353931316310824,
 'max_depth': 7,
 'min_child_weight': 3,
 'n_estimators': 496,
 'reg_alpha': 0.003212780953181041,
 'reg_lambda': 6.612146513729712,
 'subsample': 0.9594600718236017}


In [6]:
# Final fit with best params
final_pipeline, fit_time = final_fit_reg(
    preprocessor,
    best_search_params,
    x_train_full,
    y_train_full,
    device=xgb_device,
)

In [7]:
# Evaluate on train and test
train_metrics = evaluate_reg(final_pipeline, x_train_full, y_train_full)
test_metrics = evaluate_reg(final_pipeline, x_test, y_test)

In [8]:
# Final results
print("\nFinal results:")

print("Search time (seconds):", search_time)
print("Final training time (seconds):", fit_time)
print("Best CV MSE:", best_cv_mse)

print("\nTrain RMSE:", train_metrics["rmse"])
print(
    "Train inference time per sample (seconds):",
    train_metrics["infer_time_per_sample_s"],
)

print("\nTest RMSE:", test_metrics["rmse"])
print(
    "Test inference time per sample (seconds):", test_metrics["infer_time_per_sample_s"]
)


Final results:
Search time (seconds): 95.37194133199955
Final training time (seconds): 0.4961800940000103
Best CV MSE: 323749.2375

Train RMSE: 388.5367783427973
Train inference time per sample (seconds): 5.506806173211625e-07

Test RMSE: 562.1490571903505
Test inference time per sample (seconds): 6.948093252020315e-07


In [9]:
# Export model and results
export_results_reg(
    data_name,
    final_pipeline,
    best_model_params,
    train_metrics,
    test_metrics,
    fit_time,
    search_time,
    best_cv_mse,
)


Model saved to /home/shado/college_project/comp9417/9417GroupProject/models/diamonds/xgboost.joblib
Pipeline saved to /home/shado/college_project/comp9417/9417GroupProject/models/diamonds/xgboost_pipeline.joblib
Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/diamonds/xgboost.json
